# HFSS Sampled Viewer

이 노트북은 sampled/build 결과만 읽는 thin manual viewer다. 샘플링과 빌드는 notebook이 아니라 `entry/sample_type2.py`와 `entry/build_type2.py`가 owner다.

- sample owner: `entry/sample_type2.py`
- build owner: `entry/build_type2.py`
- canonical sampled root: `run/sampled/type2/<design_id>/`
- inspected artifacts: `sampled.toml`, `type2_step_ledger.json`, `type2_imported_ledger.json`, `<design_id>.aedt`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from pprint import pprint
import subprocess
import sys
import tomllib

repo_root_result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"],
    check=True,
    capture_output=True,
    text=True,
)
repo_root_text = repo_root_result.stdout.strip()
if repo_root_text == "":
    raise RuntimeError("git rev-parse --show-toplevel returned empty stdout")
REPO_ROOT = Path(repo_root_text).resolve()
pyproject_path = REPO_ROOT / "pyproject.toml"
if not pyproject_path.is_file():
    raise FileNotFoundError(f"repo root is missing pyproject.toml: {pyproject_path}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SAMPLED_ROOT = REPO_ROOT / "run" / "sampled" / "type2"
if not SAMPLED_ROOT.is_dir():
    raise FileNotFoundError(f"sampled root not found: {SAMPLED_ROOT}")
design_dirs = sorted(
    (path for path in SAMPLED_ROOT.iterdir() if path.is_dir()),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if len(design_dirs) == 0:
    raise FileNotFoundError(
        f"No sampled design directories found under {SAMPLED_ROOT}. Run entry/sample_type2.py and entry/build_type2.py first."
    )

DESIGN_DIR = design_dirs[0]
SAMPLED_TOML_PATH = DESIGN_DIR / "sampled.toml"
STEP_LEDGER_PATH = DESIGN_DIR / "type2_step_ledger.json"
IMPORTED_LEDGER_PATH = DESIGN_DIR / "type2_imported_ledger.json"
aedt_paths = tuple(DESIGN_DIR.glob("*.aedt"))
if len(aedt_paths) != 1:
    raise FileNotFoundError(
        f"Expected exactly one .aedt file in {DESIGN_DIR}, found {len(aedt_paths)}. Run entry/build_type2.py first."
    )
OUTPUT_AEDT_PATH = aedt_paths[0]

sampled_payload = tomllib.loads(SAMPLED_TOML_PATH.read_text(encoding="utf-8"))
sampled_metadata = sampled_payload["sampled"]
summary = {
    "design_dir": str(DESIGN_DIR),
    "design_id": sampled_metadata["design_id"],
    "seed": sampled_metadata["seed"],
    "source_toml_path": sampled_metadata["source_toml_path"],
    "sampled_toml_path": str(SAMPLED_TOML_PATH),
    "step_ledger_path": str(STEP_LEDGER_PATH),
    "imported_ledger_path": str(IMPORTED_LEDGER_PATH),
    "aedt_path": str(OUTPUT_AEDT_PATH),
}
pprint(summary)


## 1. Inspect Sampled TOML

sampled metadata와 frozen sampled owner를 요약한다. notebook은 sampling/build logic를 다시 계산하지 않는다.


In [ ]:
sampled_summary = {
    "metadata": sampled_payload["sampled"],
    "modeled_objects": [
        {
            "object_id": entry["object_id"],
            "sampled_ranges": {
                key: value["range"]
                for key, value in entry.items()
                if isinstance(value, dict) and set(value.keys()) == {"range"}
            },
        }
        for entry in sampled_payload["modeled_objects"]
    ],
}
pprint(sampled_summary)


## 2. Inspect Import Handoff Ledger

build runtime이 쓴 imported ledger를 읽어서 imported ownership을 요약한다. notebook은 HFSS import/setup를 호출하지 않는다.


In [ ]:
payload = json.loads(IMPORTED_LEDGER_PATH.read_text(encoding="utf-8"))

import_summary = {
    "aedt_path": payload["aedt_path"],
    "source_step_ledger_path": payload["source_step_ledger_path"],
    "scene_step_path": payload["scene_step_path"],
    "non_model_count": len(payload["non_model_objects"]),
    "modeled_count": len(payload["modeled_objects"]),
}
pprint(import_summary)

for group_name in ("non_model_objects", "modeled_objects"):
    print()
    print(group_name)
    print("-" * len(group_name))
    for entry in payload[group_name]:
        print(f"object_id: {entry['object_id']}")
        print(f"  role: {entry['role']}")
        print(f"  model_state: {entry['model_state']}")
        print(f"  imported_object_names: {entry['imported_object_names']}")
